In [ ]:
import sys
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from scipy.stats import spearmanr
from tqdm import tqdm

from utils import kernel_mean_quantile_band_rankx

sys.path.append("external/heart_failure_detection")
sys.path.append("external/heart_failure_detection/src/model")
from external.heart_failure_detection.src.model.inception_ensemble import \
    InceptionEnsemble
from external.heart_failure_detection.src.transform.transform import Normalize

weights_dir = "external/heart_failure_detection/weights"
model = InceptionEnsemble(weights_dir=weights_dir)
normalizer = Normalize()
rng = np.random.default_rng(42)

In [ ]:
ECHONEXT_BASE_PATH = "/dataset/physionet.org/files/echonext/1.1.0/"

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 512

split_files = {
    "no_split": "EchoNext_no_split_waveforms.npy",
    "train": "EchoNext_train_waveforms.npy",
    "val": "EchoNext_val_waveforms.npy",
    "test": "EchoNext_test_waveforms.npy",
}

In [ ]:
metadata = pd.read_csv(ECHONEXT_BASE_PATH + "echonext_metadata_100k.csv")
model.eval().to(device)


def load_waveforms(path: str, mmap_mode: str = "r"):
    arr = np.load(path, mmap_mode=mmap_mode)
    return arr


def preprocess_waveforms(raw_waveforms: np.ndarray, target_len: int = 5000):
    waveforms = np.transpose(raw_waveforms, (0, 3, 2, 1))
    # limb leads are linearly dependent, keep only last 8 leads (aVL, aVF, and V1-V6)
    waveforms = waveforms[:, -8:, :, 0]

    n_samples, n_leads, orig_len = waveforms.shape
    x_old = np.arange(orig_len)
    x_new = np.linspace(0, orig_len - 1, target_len)
    resampled = np.zeros((n_samples, n_leads, target_len), dtype=np.float32)

    for i in tqdm(range(n_samples), desc="Resampling ECGs"):
        for j in range(n_leads):
            resampled[i, j, :] = np.interp(x_new, x_old, waveforms[i, j, :])

    return resampled  # (N, 8, target_len)


def normalize_waveforms(waveforms_np: np.ndarray, normalizer):
    waveforms = torch.tensor(waveforms_np, dtype=torch.float32)
    waveforms = normalizer(waveforms.permute(0, 2, 1)).permute(0, 2, 1)
    return waveforms


def predict_waveforms(
    waveforms: torch.Tensor, model, batch_size: int = 1024, device: str = "cuda"
):
    predictions_list = []
    n_samples = waveforms.shape[0]

    with torch.no_grad():
        for start in tqdm(range(0, n_samples, batch_size), desc="Running inference"):
            end = start + batch_size
            batch = waveforms[start:end].to(device, non_blocking=True)
            batch_pred = model(batch)  # e.g. (B, 5, 1)
            predictions_list.append(batch_pred.cpu())

    predictions = torch.cat(predictions_list, dim=0)
    return predictions


def process_split(
    split_name: str,
    filename: str,
    base_path: str,
    model,
    normalizer,
    batch_size: int,
    device: str,
):
    print(f"===== Processing {split_name} =====")
    raw = load_waveforms(base_path + filename, mmap_mode="r")
    wf_np = preprocess_waveforms(raw, target_len=5000)
    wf_tensor = normalize_waveforms(wf_np, normalizer)  # still on CPU
    preds = predict_waveforms(wf_tensor, model, batch_size, device)
    return preds


all_predictions = {}
for split, fname in split_files.items():
    preds = process_split(
        split_name=split,
        filename=fname,
        base_path=ECHONEXT_BASE_PATH,
        model=model,
        normalizer=normalizer,
        batch_size=batch_size,
        device=device,
    )
    all_predictions[split] = preds  # torch tensor on CPU

In [ ]:
CONFIG_PATH = Path("echo_settings.yaml")

with CONFIG_PATH.open("r") as f:
    cfg = yaml.safe_load(f)

PLOT_LIMS = cfg["PLOT_LIMS"]
N_BOOTSTRAP = cfg["N_BOOTSTRAP"]

In [ ]:
def attach_predictions_to_metadata(
    metadata: pd.DataFrame, preds, prob_col: str = "ensemble"
):
    if hasattr(preds, "detach"):
        preds = preds.detach().cpu()
    p = torch.sigmoid(preds).mean(axis=1).numpy()  # average over ensemble members
    meta = metadata.copy()
    meta[prob_col] = p
    return meta


def plot_rankx_for_split(
    metadata: pd.DataFrame,
    prob_col: str = "ensemble",
    plot_cols=("lvef_value", "pasp_value", "ivs_measurement", "lvpw_measurement"),
    column_name_map: dict = None,
    column_units_map: dict = None,
    lims=None,
    n_grid: int = 400,
    qs_inner: float = 0.25,
):
    """
    Create one figure for a given split, with one panel per echo parameter.
    """
    if lims is None:
        lims = {}

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(12, 2.6665), squeeze=False)
    axs = axs[0]  # since 1 row

    legend_handles, legend_labels = None, None

    for i, (ax, column) in enumerate(zip(axs, plot_cols)):
        sub = metadata.copy()
        sub = sub[~sub[column].isna()].reset_index(drop=True)

        x = sub[column].values
        y = sub[prob_col].values

        if column == "lvef_value":
            x = x + (np.random.uniform(size=x.shape) - 0.5) * 5
        elif column == "pasp_value":
            x = x + (np.random.uniform(size=x.shape) - 0.5) * 1
        else:
            x = x + (np.random.uniform(size=x.shape) - 0.5) * 0.1

        ind = np.argsort(x)
        x = x[ind]
        y = y[ind]

        xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
            x, y, qs=(qs_inner, 1 - qs_inner), n_grid=n_grid, const=1.0
        )
        ax.plot(xg_q, qmedian, linewidth=2.0, color="C0", zorder=2, label="Median")
        ax.fill_between(
            xg_q,
            qlo,
            qhi,
            alpha=0.4,
            color="C0",
            zorder=1,
            label=f"{int(qs_inner*100)}--{int((1-qs_inner)*100)}% range",
            edgecolor="none",
        )

        ax.set_ylim(0, 1)
        if column_name_map is not None and column in column_name_map:
            ax.set_title(column_name_map[column], fontsize=13)
        else:
            ax.set_title(column.replace("_", " "), fontsize=13)
        if column_units_map is not None and column in column_units_map:
            ax.set_xlabel(f"{column_units_map[column]}")
        else:
            ax.set_xlabel(column.replace("_", " "))

        if i == 0:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
        else:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

        # ----- KDE density on twin axis -----
        ax2 = ax.twinx()

        sns.kdeplot(
            x,
            ax=ax2,
            color="gray",
            fill=True,
            alpha=0.3,
            edgecolor="gray",
            linewidth=0.0,
            label="Parameter density",
            zorder=0,
            bw_adjust=0.7,
        )

        ax2.set_yticks([])
        ax2.patch.set_visible(False)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)

        # remove spines of the histogram axis
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)
        ax2.spines["bottom"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.set_ylabel("")

        # main axis spines
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()
            legend_handles += handles2
            legend_labels += labels2

        colname = column_name_map.get(column, column)
        if PLOT_LIMS.get(colname) is not None:
            ax.set_xlim(PLOT_LIMS[colname])
            ax2.set_xlim(PLOT_LIMS[colname])
        else:
            lo, hi = np.percentile(x, [1, 99])
            ax.set_xlim(lo, hi)
            ax2.set_xlim(lo, hi)

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=3,
            frameon=False,
            fontsize=12,
        )

    fig.text(
        0.0,
        0.5,
        "Model-predicted risk of HF (%)",
        va="center",
        rotation="vertical",
        fontsize=13,
    )
    plt.tight_layout(rect=[0.00, 0.01, 1, 0.9])

    return fig


plot_cols = (
    "lvef_value",
    "pasp_value",
    "tr_max_velocity_value",
    "ivs_measurement",
    "lvpw_measurement",
)

train_metadata = metadata[metadata["split"] == "train"].reset_index(drop=True)
val_metadata = metadata[metadata["split"] == "val"].reset_index(drop=True)
test_metadata = metadata[metadata["split"] == "test"].reset_index(drop=True)
no_split_metadata = metadata[metadata["split"] == "no_split"].reset_index(drop=True)


column_name_map = {
    "lvef_value": "LVEF",
    "pasp_value": "PASP",
    "ivs_measurement": "IVS",
    "lvpw_measurement": "PWT",
    "tr_max_velocity_value": "TRv",
}
column_units_map = {
    "lvef_value": "%",
    "pasp_value": "mmHg",
    "ivs_measurement": "cm",
    "lvpw_measurement": "cm",
    "tr_max_velocity_value": "m/s",
}
all_metadata = attach_predictions_to_metadata(
    metadata,
    torch.cat(
        [
            all_predictions["train"],
            all_predictions["val"],
            all_predictions["test"],
            all_predictions["no_split"],
        ],
        dim=0,
    ),
    prob_col="ensemble",
)
fig = plot_rankx_for_split(
    all_metadata.query("most_recent_ecg == 1"),
    plot_cols=plot_cols,
    column_name_map=column_name_map,
    column_units_map=column_units_map,
)
plt.savefig("figures/png/rankx_kernel_columbia.png", dpi=300)
plt.savefig("figures/pgf/rankx_kernel_columbia.pgf", bbox_inches="tight")
plt.show()

In [ ]:
def plot_rankx_for_split_by_sex(
    metadata: pd.DataFrame,
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    male_value: str = "male",
    female_value: str = "female",
    plot_cols=("lvef_value", "pasp_value", "ivs_measurement", "lvpw_measurement"),
    column_name_map: dict = None,
    column_units_map: dict = None,
    lims=None,
    n_grid: int = 400,
    qs_inner: float = 0.45,  # for median band
    qs_outer: float = 0.25,  # for wider band
):
    """
    Like plot_rankx_for_split, but draws two smooth curves and bands
    (male and female) in the same axes.
    """
    if lims is None:
        lims = {}

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.5), squeeze=False)
    axs = axs[0]  # single row

    legend_handles, legend_labels = None, None

    # Colors per sex
    sex_specs = [
        (female_value, "C1", "Female"),
        (male_value, "C0", "Male"),
    ]

    for i, (ax, column) in enumerate(zip(axs, plot_cols)):
        sub = metadata.copy()
        sub = sub[~sub[column].isna()].reset_index(drop=True)

        if sub.empty:
            ax.set_visible(False)
            continue

        # Base x (all patients) for x-limits and density
        x_all = sub[column].values

        # jitter by column (same jitter used both sexes so they stay aligned)
        if column == "lvef_value":
            jitter = (np.random.uniform(size=x_all.shape) - 0.5) * 5
        elif column == "pasp_value":
            jitter = (np.random.uniform(size=x_all.shape) - 0.5) * 1
        else:
            jitter = (np.random.uniform(size=x_all.shape) - 0.5) * 0.1

        x_all_j = x_all + jitter

        # x-limits
        if lims.get(column) is not None:
            ax.set_xlim(lims[column])
        else:
            lo, hi = np.percentile(x_all_j, [1, 99])
            ax.set_xlim(lo, hi)

        # ----- Sex-specific curves and bands -----
        for sex_val, color, sex_label in sex_specs:
            sub_sex = sub[sub[sex_col] == sex_val]
            if sub_sex.empty:
                continue

            idx_sex = sub.index.isin(sub_sex.index)
            x = x_all_j[idx_sex]
            y = sub.loc[idx_sex, prob_col].values

            ind = np.argsort(x)
            x = x[ind]
            y = y[ind]

            xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
                x, y, qs=(qs_inner, 1 - qs_inner), n_grid=n_grid, const=1.0
            )
            ax.plot(
                xg_q,
                qmedian,
                linewidth=2.0,
                color=color,
                zorder=3,
                label=f"{sex_label} median",
            )
            ax.fill_between(
                xg_q,
                qlo,
                qhi,
                alpha=0.25,
                color=color,
                zorder=2,
                label=f"{sex_label} {int(qs_outer*100)}–{int((1-qs_outer)*100)}% range",
                edgecolor="none",
            )

        ax.set_ylim(0, 1)

        # Titles / x-labels
        if column_name_map is not None and column in column_name_map:
            ax.set_title(column_name_map[column])
        else:
            ax.set_title(column.replace("_", " "))

        if column_units_map is not None and column in column_units_map:
            ax.set_xlabel(column_units_map[column])
        else:
            ax.set_xlabel(column.replace("_", " "))

        if i == 0:
            ax.set_yticks(
                [0, 0.25, 0.5, 0.75, 1.0],
                ["0", "25", "50", "75", "100"],
            )
        else:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

        ax2 = ax.twinx()
        sns.kdeplot(
            x_all_j,
            ax=ax2,
            color="gray",
            fill=True,
            alpha=0.3,
            edgecolor="gray",
            linewidth=0.0,
            label="Parameter density",
            zorder=0,
            bw_adjust=0.5,
        )

        ax2.set_yticks([])
        ax2.patch.set_visible(False)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)

        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)
        ax2.spines["bottom"].set_visible(False)
        ax2.spines["left"].set_visible(False)
        ax2.set_ylabel("")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
            handles2, labels2 = ax2.get_legend_handles_labels()
            legend_handles += handles2
            legend_labels += labels2

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=5,
            frameon=False,
            fontsize=11,
        )

    fig.text(
        0.0,
        0.5,
        "Model-predicted risk of HF (%)",
        va="center",
        rotation="vertical",
        fontsize=11,
    )
    plt.tight_layout(rect=[0.00, 0.01, 1, 0.9])

    return fig


fig = plot_rankx_for_split_by_sex(
    all_metadata.query("most_recent_ecg == 1"),
    prob_col="ensemble",
    sex_col="sex",
    male_value="male",
    female_value="female",
    plot_cols=plot_cols,
    column_name_map=column_name_map,
    column_units_map=column_units_map,
)
plt.savefig("figures/png/rankx_kernel_columbia_male_female.png", dpi=300)
plt.savefig("figures/pgf/rankx_kernel_columbia_male_female.pgf", bbox_inches="tight")
plt.show()

In [ ]:
def _spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    return float(np.abs(spearmanr(x, y).correlation))


def plot_rankx_by_race_and_sex(
    metadata: pd.DataFrame,
    plot_cols: List[str],
    races: Optional[List[str]] = None,
    race_col: str = "race_ethnicity",
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    sexes: Sequence[str] = ("Female", "Male"),
    column_name_map: Optional[Dict[str, str]] = None,
    column_units_map: Optional[Dict[str, str]] = None,
    qs_inner: float = 0.975,
    qs_outer: float = 0.025,
    n_boot: int = 100,
    random_state: Optional[int] = None,
    min_n: int = 10,
    stats_path: Union[str, Path] = "./cache/rankx_ci_by_sex_ahus.csv",
    all_label: str = "All",
    whitespace_gap: float = 0.5,
) -> Tuple[plt.Figure, np.ndarray]:
    if not (0.0 <= qs_outer < qs_inner <= 1.0):
        raise ValueError(
            f"Invalid CI quantiles: qs_outer={qs_outer}, qs_inner={qs_inner}. "
            "Require 0 <= qs_outer < qs_inner <= 1."
        )

    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}

    # races
    if races is None:
        races = metadata[race_col].dropna().astype(str).value_counts().index.tolist()
    races = sorted(races)

    race_tick_labels = [str(r).capitalize() for r in races]

    stats_path = Path(stats_path)
    stats_df = pd.read_csv(stats_path)

    required = {
        "variable",
        "sex",
        "estimate",
        "ci_low",
        "ci_high",
        "qs_outer",
        "qs_inner",
    }
    missing = required - set(stats_df.columns)
    if missing:
        raise ValueError(f"stats_path is missing required columns: {sorted(missing)}")

    # Check CI percentiles match (exactly, within tolerance)
    q_outer_vals = stats_df["qs_outer"].dropna().unique()
    q_inner_vals = stats_df["qs_inner"].dropna().unique()
    if len(q_outer_vals) != 1 or len(q_inner_vals) != 1:
        raise ValueError(
            "stats_path has inconsistent qs_outer/qs_inner values: "
            f"qs_outer={q_outer_vals}, qs_inner={q_inner_vals}"
        )

    file_qs_outer = float(q_outer_vals[0])
    file_qs_inner = float(q_inner_vals[0])

    if not (
        np.isclose(file_qs_outer, qs_outer) and np.isclose(file_qs_inner, qs_inner)
    ):
        raise ValueError(
            f"CI percentile mismatch: function expects qs_outer={qs_outer}, qs_inner={qs_inner}, "
            f"but stats file has qs_outer={file_qs_outer}, qs_inner={file_qs_inner}."
        )

    # Optional: check n_boot if present
    if "n_boot" in stats_df.columns:
        n_boot_vals = stats_df["n_boot"].dropna().unique()
        if len(n_boot_vals) == 1:
            file_n_boot = int(n_boot_vals[0])
            if file_n_boot != int(n_boot):
                raise ValueError(
                    f"n_boot mismatch: function n_boot={n_boot} but stats file n_boot={file_n_boot}."
                )

    categories = [all_label] + races
    tick_labels = [all_label] + race_tick_labels

    x_positions = np.concatenate(
        ([0.0], (np.arange(len(races), dtype=float) + 1.0 + float(whitespace_gap)))
    )

    n_sexes = len(sexes)
    width = 0.25
    if n_sexes == 2:
        offsets = np.array([-width / 2, width / 2])
    else:
        offsets = np.linspace(-width, width, n_sexes)

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(14, 3.2), squeeze=False)
    axs = axs[0]

    legend_handles, legend_labels = None, None

    for ax_i, (ax, column) in enumerate(zip(axs, plot_cols)):
        medians = np.full((n_sexes, len(categories)), np.nan, dtype=float)
        los = np.full_like(medians, np.nan)
        his = np.full_like(medians, np.nan)

        for s_idx, sex in enumerate(sexes):
            for r_idx, race in enumerate(races):
                sex = sex.lower()
                sub = (
                    metadata.loc[
                        (metadata[race_col] == race) & (metadata[sex_col] == sex),
                        [column, prob_col],
                    ]
                    .dropna()
                    .reset_index(drop=True)
                )

                x = sub[column].to_numpy()
                y = sub[prob_col].to_numpy()
                n = len(x)

                if n < min_n:
                    continue

                boot_corrs = np.empty(n_boot, dtype=float)
                for b in range(n_boot):
                    idx = rng.integers(0, n, size=n)
                    boot_corrs[b] = _spearman_corr(x[idx], y[idx])

                boot_corrs = boot_corrs[~np.isnan(boot_corrs)]
                if boot_corrs.size == 0:
                    continue

                j = 1 + r_idx
                medians[s_idx, j] = np.median(boot_corrs)
                los[s_idx, j] = np.quantile(boot_corrs, qs_outer)
                his[s_idx, j] = np.quantile(boot_corrs, qs_inner)

        for s_idx, sex in enumerate(sexes):
            sex = sex.lower()
            colname = column_name_map.get(column, column)
            row = stats_df.loc[
                (stats_df["variable"] == colname) & (stats_df["sex"] == sex)
            ]
            if row.empty:
                continue
            row0 = row.iloc[0]
            medians[s_idx, 0] = float(row0["estimate"])
            los[s_idx, 0] = float(row0["ci_low"])
            his[s_idx, 0] = float(row0["ci_high"])

        for s_idx, sex in enumerate(sexes):
            sex_medians = medians[s_idx]
            sex_los = los[s_idx]
            sex_his = his[s_idx]

            valid = ~np.isnan(sex_medians)
            if not np.any(valid):
                continue

            yerr = np.vstack(
                [
                    sex_medians[valid] - sex_los[valid],
                    sex_his[valid] - sex_medians[valid],
                ]
            )

            ax.errorbar(
                x_positions[valid] + offsets[s_idx],
                sex_medians[valid],
                yerr=yerr,
                fmt="o",
                capsize=4,
                label=str(sex).capitalize(),
                color=f"C{3 - s_idx * 3}",
            )

        ax.set_xticks(x_positions)
        ax.set_xticklabels(tick_labels, rotation=0, ha="center", fontsize=9)
        ax.set_ylabel(r"Spearman |$\rho$|", fontsize=12)

        x_min = float(np.min(x_positions))
        x_max = float(np.max(x_positions))
        ax.set_xlim(x_min - 0.75, x_max + 0.75)

        pretty = column_name_map.get(column, column)
        ax.set_title(pretty, fontsize=13)

        ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4, 0.5])
        ax.set_ylim(0, 0.55)
        if ax is not axs[0]:
            ax.set_yticklabels([])
            ax.set_ylabel("")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        secax = ax.secondary_xaxis("bottom")
        secax.set_xticks(
            [
                x_positions[0],
                np.mean(x_positions[1:]),
            ]
        )
        secax.set_xticklabels(["Ahus", "CUIMC"], fontsize=9)

        secax.spines["bottom"].set_position(("outward", 18))
        secax.spines["bottom"].set_visible(False)

        secax.tick_params(axis="x", length=0, pad=5)

        if ax_i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()
        ax.set_xlim(-0.5, 5)

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=min(len(legend_labels), 5),
            frameon=False,
            fontsize=12,
        )

    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.90])
    return fig, axs

In [ ]:
races = ["black", "white", "asian", "hispanic"]

fig, axs = plot_rankx_by_race_and_sex(
    all_metadata.query(
        "most_recent_ecg == 1 and race_ethnicity != 'unknown' and race_ethnicity != 'other'"
    ),
    plot_cols=plot_cols,
    races=races,
    race_col="race_ethnicity",
    prob_col="ensemble",
    sex_col="sex",
    sexes=("Female", "Male"),
    column_name_map=column_name_map,
    column_units_map=column_units_map,
    qs_inner=0.975,
    qs_outer=0.025,
    n_boot=N_BOOTSTRAP,
    min_n=10,
)
plt.savefig("figures/png/median_spearman_by_race_and_site.png", dpi=300)
plt.savefig("figures/pgf/median_spearman_by_race_and_site.pgf", bbox_inches="tight")
plt.show()

In [ ]:
def plot_rankx_by_sex_and_race(
    metadata: pd.DataFrame,
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    race_col: str = "race_ethnicity",
    male_value: str = "male",
    female_value: str = "female",
    races: Optional[List[str]] = None,
    sexes: Sequence[str] = ("female", "male"),
    plot_cols=("lvef_value", "pasp_value", "ivs_measurement", "lvpw_measurement"),
    column_name_map: Dict[str, str] = None,
    column_units_map: Dict[str, str] = None,
    lims=None,
    n_grid: int = 400,
    qs_median_band: float = 0.25,
):
    """
    Stratify by both sex and race_ethnicity.

    - Two rows: one per sex (first females, then males).
    - Columns are measurement variables (plot_cols).
    - Color encodes race.
    - Only a single smooth 'median' curve per race group (no IQR bands).
    - KDEs are separate per sex (row).
    """

    if lims is None:
        lims = {}
    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}

    # Determine race order if not provided
    if races is None:
        races = metadata[race_col].dropna().astype(str).value_counts().index.tolist()
    races = sorted(races)

    race_colors = {race: f"C{i % 10}" for i, race in enumerate(races)}

    n_cols = len(plot_cols)
    n_rows = len(sexes)

    fig, axs = plt.subplots(
        n_rows,
        n_cols,
        figsize=(12, 4.7),
        squeeze=False,
    )

    legend_entries = {}

    for j, column in enumerate(plot_cols):
        sub_all = metadata[~metadata[column].isna()].reset_index(drop=True)
        if sub_all.empty:
            for i in range(n_rows):
                axs[i, j].set_visible(False)
            continue

        x_all = sub_all[column].values

        if column == "lvef_value":
            jitter_all = (np.random.uniform(size=x_all.shape) - 0.5) * 5
        elif column == "pasp_value":
            jitter_all = (np.random.uniform(size=x_all.shape) - 0.5) * 1
        else:
            jitter_all = (np.random.uniform(size=x_all.shape) - 0.5) * 0.1

        x_all_j_all = x_all + jitter_all

        if lims.get(column) is not None:
            x_lo, x_hi = lims[column]
        else:
            x_lo, x_hi = np.percentile(x_all_j_all, [1, 99])

        for i, sex in enumerate(sexes):
            ax = axs[i, j]

            sub = sub_all[sub_all[sex_col] == sex].reset_index(drop=True)
            if sub.empty:
                ax.set_visible(False)
                continue

            x_sex = sub[column].values
            if column == "lvef_value":
                jitter = (np.random.uniform(size=x_sex.shape) - 0.5) * 5
            elif column == "pasp_value":
                jitter = (np.random.uniform(size=x_sex.shape) - 0.5) * 1
            else:
                jitter = (np.random.uniform(size=x_sex.shape) - 0.5) * 0.1

            x_sex_j = x_sex + jitter

            ax.set_xlim(x_lo, x_hi)

            for race in races:
                sub_race = sub[sub[race_col] == race]
                if sub_race.empty:
                    continue

                # print the number of non-nan values

                color = race_colors[race]

                idx_race = (sub[race_col] == race).values
                x = x_sex_j[idx_race]
                y = sub.loc[idx_race, prob_col].values

                ind = np.argsort(x)
                x = x[ind]
                y = y[ind]

                xg_q, qmean, qmedian, qlo, qhi = kernel_mean_quantile_band_rankx(
                    x,
                    y,
                    qs=(qs_median_band, 1 - qs_median_band),
                    n_grid=n_grid,
                    const=1.2,
                )

                label = race

                line = ax.plot(
                    xg_q,
                    qmedian,
                    linewidth=2.0,
                    color=color,
                    linestyle="-",
                    zorder=3,
                    label=label,
                )[0]

                if label not in legend_entries:
                    legend_entries[label] = line

            ax.set_ylim(0, 1)

            if column in column_name_map:
                ax.set_title(column_name_map[column], fontsize=13)
            else:
                ax.set_title(column.replace("_", " "), fontsize=13)

            if column in column_units_map:
                ax.set_xlabel(column_units_map[column], fontsize=11)
            else:
                ax.set_xlabel(column.replace("_", " "), fontsize=11)

            if j == 0:
                ax.set_yticks(
                    [0, 0.25, 0.5, 0.75, 1.0],
                    ["0", "25", "50", "75", "100"],
                )
            else:
                ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

            if j == 0:
                ax.text(
                    -0.25,
                    0.5,
                    sex.capitalize(),
                    transform=ax.transAxes,
                    rotation=90,
                    va="center",
                    ha="center",
                    fontsize=11,
                )

            ax2 = ax.twinx()
            sns.kdeplot(
                x_sex_j,
                ax=ax2,
                color="gray",
                fill=True,
                alpha=0.3,
                edgecolor="gray",
                linewidth=0.0,
                zorder=0,
                bw_adjust=0.6,
            )

            ax2.set_yticks([])
            ax2.patch.set_visible(False)
            ax.set_zorder(ax2.get_zorder() + 1)
            ax.patch.set_visible(False)

            ax2.spines["top"].set_visible(False)
            ax2.spines["right"].set_visible(False)
            ax2.spines["bottom"].set_visible(False)
            ax2.spines["left"].set_visible(False)
            ax2.set_ylabel("")
            colname = column_name_map.get(column, column)
            if PLOT_LIMS.get(colname) is not None:
                ax.set_xlim(PLOT_LIMS[colname])

    if legend_entries:
        handles = list(legend_entries.values())
        labels = list(legend_entries.keys())
        # capitalize labels
        labels = [lbl.capitalize() for lbl in labels]
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=min(len(labels), 4),
            frameon=False,
            fontsize=12,
        )

    fig.text(
        0.0,
        0.5,
        "Model-predicted risk of HF (%)",
        va="center",
        rotation="vertical",
        fontsize=12,
    )

    for row_axes in axs:
        for ax in row_axes:
            if not ax.get_visible():
                continue
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

    for j in range(n_cols):
        ax_top = axs[0, j]
        ax_bottom = axs[1, j]
        if ax_top.get_visible():
            ax_top.set_xlabel("")
            ax_top.set_xticklabels([])
        if ax_bottom.get_visible():
            ax_bottom.set_title("")
    plt.tight_layout(rect=[0.015, 0.00, 1, 0.94])

    return fig


fig = plot_rankx_by_sex_and_race(
    all_metadata.query(
        "most_recent_ecg == 1 and race_ethnicity != 'unknown' and race_ethnicity != 'other'"
    ),
    prob_col="ensemble",
    sex_col="sex",
    race_col="race_ethnicity",
    male_value="male",
    female_value="female",
    races=None,
    sexes=("male", "female"),
    plot_cols=plot_cols,
    column_name_map=column_name_map,
    column_units_map=column_units_map,
)

plt.savefig("figures/png/rankx_kernel_by_race_sex_columbia.png", dpi=300)
plt.savefig("figures/pgf/rankx_kernel_by_race_sex_columbia.pgf", bbox_inches="tight")
plt.show()

In [ ]:
# Build the counts table
counts = (
    all_metadata.query("most_recent_ecg == 1")
    .dropna(subset=["race_ethnicity", "sex"])
    .groupby(["race_ethnicity", "sex"])
    .size()
    .reset_index(name="n_patients")
)

table = (
    counts.pivot(index="race_ethnicity", columns="sex", values="n_patients")
    .fillna(0)
    .astype(int)
)

table.columns = [col.capitalize() for col in table.columns]

table["Total"] = table.sum(axis=1)

ordered_cols = ["Total"] + [c for c in ["Male", "Female"] if c in table.columns]
table = table[ordered_cols]

table = table.sort_values(by="Total", ascending=False)

lines = []

lines.append("\\begin{table}")
lines.append("    \\centering")
lines.append("    \\begin{threeparttable}")
lines.append("    \\caption{Counts of patients by race and sex.}")
lines.append("    \\label{tab:race_sex_counts}")

col_spec = "l" + "r" * len(table.columns)

lines.append(r"    {\setlength{\tabcolsep}{17pt}")
lines.append("    \\begin{tabular}{" + col_spec + "}")
lines.append("        \\toprule")

header = "Race"
for col in table.columns:
    header += " & " + str(col)
header += " \\\\"
lines.append("        " + header)
lines.append("        \\midrule")

for race in table.index:
    row = race.capitalize()
    for col in table.columns:
        val = table.loc[race, col]
        row += " & \\num{" + str(val) + "}"
    row += " \\\\"
    lines.append("        " + row)

lines.append("        \\bottomrule")
lines.append("    \\end{tabular}")
lines.append("    }")  # end setlength

lines.append("    \\end{threeparttable}")
lines.append("\\end{table}")

latex_output = "\n".join(lines)

with open("tables/race_sex_counts_table.tex", "w") as f:
    f.write(latex_output)